In [9]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

In [11]:
def phase_oracle(n_qubits, target_state):
    oracle_circuit = QuantumCircuit(n_qubits, name="Oracle")
    reversed_target = target_state[::-1]
    for qubit, bit in enumerate(reversed_target):
        if bit == '0':
            oracle_circuit.x(qubit)
    oracle_circuit.h(n_qubits -1)
    oracle_circuit.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    oracle_circuit.h(n_qubits - 1)
    
    for qubit, bit in enumerate(reversed_target):
        if bit == '0':
            oracle_circuit.x(qubit)
            
    return oracle_circuit
    

In [12]:
def diffuser(n_qubits):
    diffuser_circuit = QuantumCircuit(n_qubits, name="Diffuser")
    diffuser_circuit.h(range(n_qubits))
    diffuser_circuit.x(range(n_qubits))
    
    diffuser_circuit.h(n_qubits - 1)
    diffuser_circuit.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    diffuser_circuit.h(n_qubits - 1)
    
    diffuser_circuit.x(range(n_qubits))
    
    diffuser_circuit.h(range(n_qubits))
    
    return diffuser_circuit


In [13]:
def grover_search(n_qubits, target_state):
    N = 2**n_qubits
    iterations = int(np.floor(np.pi / 4 * np.sqrt(N)))
    print(f"Searching for '{target_state}' across {n_qubits} qubits (Database size N = {N})")
    print(f"Running optimal number of iterations: {iterations}\n")
    
    grover_circuit = QuantumCircuit(n_qubits, n_qubits)
    
    grover_circuit.h(range(n_qubits))
    grover_circuit.barrier()
    
    oracle_gate = phase_oracle(n_qubits, target_state).to_gate()
    diffuser_gate = diffuser(n_qubits).to_gate()
    
    for _ in range(iterations):
        grover_circuit.append(oracle_gate, range(n_qubits))
        grover_circuit.append(diffuser_gate, range(n_qubits))
        grover_circuit.barrier()
        
    grover_circuit.measure(range(n_qubits), range(n_qubits))
    
    return grover_circuit

In [14]:
if __name__ == "__main__":
    NUM_QUBITS = 4
    TARGET = "1101"  
    circuit = grover_search(NUM_QUBITS, TARGET)
    
    print("Circuit Summary Blueprint:")
    print(circuit.draw(output='text'))
    
    simulator = AerSimulator()
    compiled_circuit = transpile(circuit, simulator) 

    result = simulator.run(compiled_circuit, shots=1024).result()
    counts = result.get_counts()

Searching for '1101' across 4 qubits (Database size N = 16)
Running optimal number of iterations: 3

Circuit Summary Blueprint:
     ┌───┐ ░ ┌─────────┐┌───────────┐ ░ ┌─────────┐┌───────────┐ ░ ┌─────────┐»
q_0: ┤ H ├─░─┤0        ├┤0          ├─░─┤0        ├┤0          ├─░─┤0        ├»
     ├───┤ ░ │         ││           │ ░ │         ││           │ ░ │         │»
q_1: ┤ H ├─░─┤1        ├┤1          ├─░─┤1        ├┤1          ├─░─┤1        ├»
     ├───┤ ░ │  Oracle ││  Diffuser │ ░ │  Oracle ││  Diffuser │ ░ │  Oracle │»
q_2: ┤ H ├─░─┤2        ├┤2          ├─░─┤2        ├┤2          ├─░─┤2        ├»
     ├───┤ ░ │         ││           │ ░ │         ││           │ ░ │         │»
q_3: ┤ H ├─░─┤3        ├┤3          ├─░─┤3        ├┤3          ├─░─┤3        ├»
     └───┘ ░ └─────────┘└───────────┘ ░ └─────────┘└───────────┘ ░ └─────────┘»
c: 4/═════════════════════════════════════════════════════════════════════════»
                                                                        